# N1 — Kinematics Notebook
**Competencies** C2–C4 · **Artifact** Geometry / IK-FK / Workspace map · **Input** Family 1 CSV exports

Reproduce IK/FK/det J from the workspace grid, analyze the reachable region, verify the IK→FK round-trip.


## 1-2. Purpose & inputs
Inputs: `B1-workspace-map.csv` (2-DOF), `B2`/`B3` (3-DOF). Geometry matches the engine: b=0.6, L∈[0.4,1.0].


In [1]:
from pathlib import Path
import csv, json, math, urllib.request
# Reference CSVs live in docs/figures. In-repo this resolves directly; on Colab the
# files are pulled from raw GitHub. Students can instead drop their own demo exports in DATA.
REPO_RAW='https://raw.githubusercontent.com/alibulentkoc/parallel-kinematics-hydraulics/main/docs/figures'
DATA = Path('../figures') if Path('../figures').exists() else Path('figures')
DATA.mkdir(exist_ok=True) if DATA==Path('figures') else None
def _ensure(name):
    p=DATA/name
    if not p.exists():
        try: urllib.request.urlretrieve(f'{REPO_RAW}/{name}', p); print('fetched', name)
        except Exception as e: raise FileNotFoundError(f'{name} not found locally and fetch failed: {e}')
    return p
def load_csv(name):
    rows=[]
    with open(_ensure(name)) as f:
        reader=csv.reader(l for l in f if not l.startswith('#'))
        header=next(reader)
        for r in reader:
            if r: rows.append(dict(zip(header,r)))
    return header, rows
def col(rows,k,cast=float):
    return [cast(r[k]) for r in rows if r.get(k) not in (None,'')]


In [2]:
# engine kinematics (reproduced from src/, not new theory)
B=0.6; LMIN,LMAX=0.4,1.0
def ik2(x,y): return [math.hypot(x+B,y), math.hypot(x-B,y)]
def det2(x,y):
    L=ik2(x,y); return abs(2*B*y/(L[0]*L[1])) if L[0]*L[1] else 0.0
def fk2(L1,L2):
    x=(L1*L1-L2*L2)/(4*B); y=math.sqrt(max(0.0,L1*L1-(x+B)**2)); return x,y


## 3. Reproduce simulator result
Recompute det J for each reachable grid cell and confirm it matches the exported `detJ` column.


In [3]:
_,b1=load_csv('B1-workspace-map.csv')
maxdiff=0.0; n=0
for r in b1:
    if r['reachable']!='1' or r['detJ']=='' : continue
    d=det2(float(r['x']),float(r['y'])); maxdiff=max(maxdiff,abs(d-float(r['detJ']))); n+=1
print(f'reproduced det J for {n} reachable cells; max |diff| vs CSV = {maxdiff:.2e}')
# CSV grid coordinates are stored to 4 dp; recomputing det J from them matches the figure to ~1e-3.
# (The tight 1e-6 threshold belongs to the full-precision IK->FK round-trip, asserted below.)
assert maxdiff < 1e-3, 'reproduction must match the exported figure within CSV coordinate rounding'


reproduced det J for 258 reachable cells; max |diff| vs CSV = 1.34e-04


## 4. Analyze


In [4]:
reach=[r for r in b1 if r['reachable']=='1']
frac=len(reach)/len(b1)
dets=[float(r['detJ']) for r in reach if r['detJ']!='']
print(f'reachable cells: {len(reach)}/{len(b1)}  ({100*frac:.1f}% of grid)')
print(f'manipulability det J: min {min(dets):.3f}  max {max(dets):.3f}  mean {sum(dets)/len(dets):.3f}')


reachable cells: 258/2116  (12.2% of grid)
manipulability det J: min 0.009  max 1.000  mean 0.768


## 5-6. Generate artifact + verify round-trip
Acceptance: IK→FK round-trip < 1e-6 m (2-DOF).


In [5]:
pose=(0.10,0.70)
L=ik2(*pose); x2,y2=fk2(*L)
err=math.hypot(x2-pose[0], y2-pose[1])
print(f'pose {pose} -> L {[round(v,3) for v in L]} -> FK ({x2:.5f},{y2:.5f})  round-trip err {err:.2e} m')
assert err < 1e-6, 'IK/FK round-trip threshold (2-DOF)'
print('PASS round-trip < 1e-6 m')


pose (0.1, 0.7) -> L [0.99, 0.86] -> FK (0.10000,0.70000)  round-trip err 5.55e-17 m
PASS round-trip < 1e-6 m


## 7. Export


In [6]:
with open('workspace-stats.csv','w',newline='') as f:
    w=csv.writer(f); w.writerow(['metric','value'])
    w.writerow(['reachable_fraction',round(frac,4)]); w.writerow(['detJ_min',round(min(dets),4)])
    w.writerow(['detJ_max',round(max(dets),4)]); w.writerow(['roundtrip_err_m',err])
print('exported workspace-stats.csv')


exported workspace-stats.csv
